# I) Reference simulation

## 1) Geometry

First, we define the geometry, material distribution, and generate the mesh.

In [ ]:
pole_pairs = 6
bundles_per_half_slot = 5

# Generate the corresponding mesh
from utils.geometry import machine_mesh
mesh = machine_mesh(p=pole_pairs, 
                    bundles_per_half_slot=bundles_per_half_slot, 
                    hBundle=0.25e-3
                    )

print(f"Generated mesh with {mesh.nv} nodes, {mesh.ne} elements")

In [ ]:
# Display the mesh
from ngsolve.webgui import Draw
print("Machine mesh")
Draw(mesh.Curve(2), filename = 'scenes/ref/mesh.html')

In [ ]:
# Display the different material distribution
air = 0
magnet = 1
iron = 2
copper = 3

zones = mesh.MaterialCF({"rotor" : magnet,
                         "core_stator" : iron,
                         "slot.*_bundle.*" : copper,
                         }, default = air)

print("Material distribution")
Draw(zones, mesh,
     settings = {"Objects" : {"Wireframe" : False}},
     filename = "scenes/ref/materials.html")

_______
## 2) Physical state computation

### 2.a) Material properties

In [ ]:
# Magnetic permeability
mur_iron = 1000

from ngsolve import pi
mu0 = 4e-7 * pi

# Define reluctivity necessary for the magnetic vector-potential formulation
nu = mesh.MaterialCF({"core_stator" : 1 / (mu0 * mur_iron)}, default = 1/mu0) 

# Display relative permeability
print("Relative permeability")
Draw(1/(mu0*nu), mesh,
     settings = {"Objects" : {"Wireframe" : False}},
     filename = "scenes/ref/mu_r.html")

In [ ]:
# Electrical conductivity

sigma_copper = 6e7
sigma = mesh.MaterialCF({"slot(.*)_bundle.*" : sigma_copper})

# Display conductivity distribution
print("Conductivity (S/m)")
Draw(sigma, mesh,
     settings = {"Objects" : {"Wireframe" : False}},
     filename = "scenes/ref/sigma.html")

In [ ]:
# Rotating remanence / magnetization

Br_magnets = 1 # remanence in T

# Define magnetization in A/m necessary for the magnetic vector-potential formulation
from utils.physics import magnetization_halbach
M = mesh.MaterialCF({"rotor" : magnetization_halbach(br = Br_magnets, p = pole_pairs)}) # A/m

# Display remanence distribution
electric_angle = 0 # angle can be changed, try 90 or 180°
print(f"Remanent flux density (T) at an electric angle of {electric_angle:.0f}°")
from ngsolve import CF, exp
Draw(mu0 * CF(((M[0]*exp(1j*pi/180*electric_angle)).real, 
               (M[1]*exp(1j*pi/180*electric_angle)).real)), mesh,
     vectors = {"grid_size": 40},
     settings = {"Objects" : {"Wireframe" : False, "Surface": False}},
     filename = f"scenes/ref/Br_{electric_angle:.0f}°el.html")

### 2.b) Current supply

In [ ]:
# Define the supply parmeters

Jrms = 10e6          # A/m²
load_angle = 0       # rad
frequency = 1000     # Hz

from ngsolve import Integrate
Irms = Jrms * Integrate(1, mesh.Materials("slot11_bundle0"))

# Distribute the current through conductors
from utils.supply import phase_current
phase = phase_current(I_rms=Irms, load_angle=load_angle)

# Display the A-B-C currents
import matplotlib.pyplot as plt
import numpy as np
from ngsolve import pi, exp
theta = np.linspace(0, 4*pi, 100)
plt.plot(theta, (phase["Ap"] * exp(1j * theta)).real, label = "A")
plt.plot(theta, (phase["Bp"] * exp(1j * theta)).real, label = "B")
plt.plot(theta, (phase["Cp"] * exp(1j * theta)).real, label = "C")
plt.xlabel("Electrical angle (rad)"); plt.ylabel("I (A)"); 
plt.grid(); plt.legend(); plt.title("3-phase current supply")
plt.show()

In [ ]:
# Define the winding arangement
winding_type = "distributed" # "concentrated"

# Distribute the current within the conductors
from utils.supply import winding_arrangement, bundle_arrangement
winding = winding_arrangement(phase = phase, type = winding_type)
bundles = bundle_arrangement(winding = winding, bundles_per_half_slot = bundles_per_half_slot)

# Display current distribution 
electric_angle = 0 # angle can be changed, try 90 or 180
print(f"Current (A) at an electric angle of {electric_angle:.0f}°")
Draw((mesh.MaterialCF(bundles)*exp(1j*pi/180*electric_angle)).real, mesh,
     settings = {"Objects" : {"Wireframe" : False}},
     filename = f"scenes/ref/J_{electric_angle:.0f}°el.html")

### 2.c) Solve magneto-harmonic problem

#### 2.b.i) Finite element space

In [ ]:
# define anti-periodic finite element space with inner and outer Dirichlet boundary conditions
curve_order = 2
fem_order = curve_order
dirichlet_bnd = "shaft|out"

from ngsolve import Periodic, H1
fes = Periodic( H1(mesh.Curve(curve_order), 
                   order = fem_order, 
                   dirichlet = dirichlet_bnd, 
                   complex = True),  [-1]*7 )

print(f"Number of degrees of freedom : {fes.ndof}")

#### 2.b.ii) Solve the problem

In [ ]:
from utils.physics import solve_magnetoharmonic

result_ref = solve_magnetoharmonic(
    fes = fes, 
    frequency = frequency,
    reluctivity = nu,
    magnetization = M,
    conductivity=sigma,
    supply = bundles)

In [ ]:
# Display the flux lines (isovalues of vector potential z-component)

electric_angle = 0
print(f"Magnetic vector potential (T.m) at an electrical angle of {electric_angle:.0f}°")
Draw((result_ref["solution"]["a"]*exp(1j *pi/180 * electric_angle)).real, 
      result_ref["info"]["fes"].mesh,
      settings = {"Objects" : {"Wireframe" : False}},
      filename = f"scenes/ref/A_{electric_angle:.0f}°el.html")

_____
## 3) Post-processing

Due to the linear magnetic material assmption, the flux density can be quite high (approximate amplitude of 2T), and is even higher at the corner due to geometrical singularities.

Considering the saturation could reduce he flux amplitude, especially at the corners.

In [ ]:
# Flux density

electric_angle = 0
from utils.physics import Curl
B = Curl(result_ref["solution"]["a"])
print("Magnetic flux density (T) at an electrical angle of {electric_angle:.0f}°")
Draw((B*exp(1j *pi/180 * electric_angle)).real,
     result_ref["info"]["fes"].mesh,
     vectors={"grid_size" : 50, "offset" : 0.5},
     settings = {"Objects" : {"Wireframe" : False}, "Colormap": {"ncolors":32}},
     filename = f"scenes/ref/B_{electric_angle:.0f}°el.html")

In [ ]:
# Norm of current density

from utils.physics import current_density
from ngsolve import Norm

j  = current_density(result_ref)
print("Current density amplitude (A/m²)")
Draw(Norm(j), result_ref["info"]["fes"].mesh,
     vectors={"grid_size" : 50, "offset" : 0.5},
     settings = {"Objects" : {"Wireframe" : False}, "Colormap": {"ncolors":32}},
     filename = f"scenes/ref/J.html")

In [ ]:
# Joule losses

from utils.physics import joule_losses

joule_losses_ref = joule_losses(result_ref)
print(f"Reference Joule losses (phi = 150°): {joule_losses_ref:.2e} W/m")

____
## 4) Optimal current angle

We look for the current angle that maximizes the torque. We make a simple exhaustive search with a 10° step size.
Since the geometry and material distribution doesn't change, we can reuse the already decomposed FEM matrix from the reference computation, so the sweep is quite fast.

In [ ]:
from utils.physics import solve_magnetoharmonic, average_torque, Curl
import numpy as np

load_angle = np.arange(0,180,10)
torque = []
scene = Draw(CF(0), fes.mesh, 
             settings = {"Objects" : {"Wireframe" : False}})
for phi in load_angle:
    phase = phase_current(I_rms=Irms,  load_angle=phi*pi/180)
    winding = winding_arrangement(phase, type = winding_type)
    bundles = bundle_arrangement(winding, bundles_per_half_slot=bundles_per_half_slot)
    result = solve_magnetoharmonic(fes=fes,
                                   reluctivity=nu,
                                   conductivity=sigma,
                                   magnetization=M,
                                   frequency=frequency,
                                   supply=bundles,
                                   Kinv=result_ref["info"]["Kinv"] # to accelerate the comutation
                                   )
    scene.Redraw(Norm(Curl(result["solution"]["a"])), result["info"]["fes"].mesh, 
                 settings = {"Objects" : {"Wireframe" : False}, "Colormap" : {"ncolors" : 32}},
                 min = 0, max = 3)
    print(f"Z-component of vector potential (Wb/m for Lz = 1m, {phi = :.0f}°/180°)"+" "*10, end = "\r")
    torque.append(average_torque(result))

In [ ]:
# Selection of the optimal angle
i_max = np.argmax(torque)
phi_max = load_angle[i_max]
torque_max = torque[i_max]

# Display
import matplotlib.pyplot as plt
plt.plot(load_angle, torque, 'o--')
plt.plot([phi_max,phi_max], [0, torque_max], 'r--')
plt.xlabel("Current angle (°)")
plt.ylabel("Average torque (Nm/m)")
plt.title(f"Average torque (max for phi = {phi_max}°)")
plt.grid(); plt.show()

We find that the torque is maximum at $\varphi = 150°$ and chose this angle for the minimization of AC losses.